# Latent Decision Gates: Highway, Hard Concrete, Sparsemax, and PonderNet (C03 Topic 12)

**Citations:** Srivastava et al. (2015 Highway Networks); Martins & Astudillo (2016 Sparsemax); Dauphin et al. (2017 GLU); Louizos et al. (2018 Hard Concrete L0); Perez et al. (2018 FiLM); Banino et al. (2021 PonderNet).

### Purpose & Topic Bridges
Natural data exhibits extreme complexity heterogeneities. Standard networks process all inputs through static computation graphs using dense activations (Sigmoid, Softmax) that never attain exact zeros, leaking noise and preventing true circuit skipping. Latent Decision Gates introduce continuous, differentiable routing, discrete feature selection, and adaptive deliberation. Highway transform gates dynamically blend nonlinear transforms with identity carry paths. Hard Concrete stretched-and-rectified gates enable exact $L_0$ feature selection with closed-form expected activity penalties. Sparsemax projects logits onto the simplex with exact zeros. PonderNet introduces geometric halting hazards for adaptive computation time.

**Decision Boundary:** Use Highway Networks / GLU when constructing very deep networks (initialize $b_T < 0$ to preserve identity carry flow). Use Hard Concrete or Sparsemax when structural sparsity or exact conditional branch execution is required. Use PonderNet with geometric prior regularization rather than Graves' ACT linear ponder penalty to prevent premature halting collapse.

## 2. Mathematical Foundations & Governing Equations

1. **Highway Network Layer:** $y = T(x) \odot H(x) + (1 - T(x)) \odot x$ where $H(x) = \text{ReLU}(x W_H + b_H)$ and $T(x) = \sigma(x W_T + b_T)$.
2. **Gated Linear Unit (GLU):** $z = x W + b = [z_a, z_b] \implies \text{GLU}(x) = z_a \odot \sigma(z_b)$, providing linear non-vanishing gradient paths.
3. **Hard Concrete $L_0$ Stochastic Gates:** $u \sim \mathcal{U}(0, 1) \implies s = \sigma((\text{logit}(u) + \log \alpha)/\beta) \implies \bar{s} = s(\zeta - \gamma) + \gamma \implies g = \min(1, \max(0, \bar{s}))$, with expected $L_0$ activity $\mathbb{P}(g > 0) = \sigma(\log \alpha - \beta \log(-\gamma / \zeta))$.
4. **Sparsemax Exact-Zero Simplex Projection:** $p = \arg\min_{q \in \Delta} \|q - z\|^2 = \max(0, z - \tau(z))$ where $\tau(z) = (\sum_{j=1}^{k^*} z_{(j)} - 1)/k^*$.
5. **PonderNet Halting Hazards:** Stopping distribution $p_n = \lambda_n \prod_{j=1}^{n-1}(1 - \lambda_j)$, expected steps $\mathbb{E}[N] = \sum n p_n$, and geometric prior penalty $D_{\text{KL}}(p \parallel \text{Geom}(\lambda_p))$.
6. **Feature-wise Linear Modulation (FiLM):** $\text{FiLM}(x; \gamma, \beta) = \gamma \odot x + \beta$ for global context conditioning.

In [ ]:
import sys, math
from pathlib import Path
import torch, torch.nn as nn

repo_root = Path.cwd().resolve()
while not (repo_root / "core").exists() and repo_root.parent != repo_root: repo_root = repo_root.parent
if str(repo_root) not in sys.path: sys.path.insert(0, str(repo_root))
import core
core.setup_env(seed=42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ckpt_path = repo_root / "weights/03_building_blocks_and_losses/12_latent_decision_gates_highway_hard_concrete_entmax_and_ponder_halting.safetensors"
weights, test_in, test_out, meta = core.parity.load_reference_checkpoint(ckpt_path, device=device)
print(f"Device: {device} | Checkpoint: {ckpt_path.name} | Loaded {len(weights)} tensors")

In [ ]:
class LatentDecisionGatesUnifiedModule(nn.Module):
    def __init__(self, in_dim=4, out_dim=4):
        super().__init__()
        self.linear = nn.Linear(in_dim, out_dim)

    def forward(self, x): return self.linear(x)

    def highway(self, x, w_h, b_h, w_t, b_t):
        h = torch.relu(x @ w_h + b_h); t = torch.sigmoid(x @ w_t + b_t)
        return t * h + (1.0 - t) * x

    def glu(self, x, w, b):
        z = x @ w + b; d = z.shape[-1] // 2
        return z[..., :d] * torch.sigmoid(z[..., d:])

    def hard_concrete(self, log_alpha, beta=0.67, gamma=-0.1, zeta=1.1, u=None):
        logit_u = torch.log(u) - torch.log(1.0 - u)
        s = torch.sigmoid((logit_u + log_alpha) / beta)
        g = torch.clamp(s * (zeta - gamma) + gamma, min=0.0, max=1.0)
        l0 = torch.sigmoid(log_alpha - beta * math.log(-gamma / zeta))
        return g, l0

    def sparsemax(self, z):
        z_s, _ = torch.sort(z, descending=True, dim=-1)
        k = torch.arange(1, z.shape[-1] + 1, device=z.device, dtype=z.dtype)
        cumsum = torch.cumsum(z_s, dim=-1)
        k_star = torch.sum(((1.0 + k * z_s) > cumsum).to(dtype=k.dtype), dim=-1, keepdim=True)
        tau = (torch.gather(cumsum, -1, (k_star - 1).long()) - 1.0) / k_star
        return torch.clamp(z - tau, min=0.0)

    def pondernet(self, lambdas, lambda_p=0.3, eps=1e-8):
        B, N = lambdas.shape; p = torch.zeros_like(lambdas); unhalted = torch.ones(B, 1, device=lambdas.device)
        for n in range(N):
            p[:, n:n+1] = lambdas[:, n:n+1] * unhalted
            unhalted = unhalted * (1.0 - lambdas[:, n:n+1])
        steps = torch.arange(1, N + 1, device=lambdas.device, dtype=lambdas.dtype).unsqueeze(0)
        exp_steps = torch.sum(p * steps, dim=-1)
        p_star = lambda_p * ((1.0 - lambda_p) ** (steps - 1.0))
        kl = torch.sum(p * (torch.log(p.clamp(min=eps)) - torch.log(p_star.clamp(min=eps))), dim=-1)
        return p, exp_steps, kl

    def film(self, x, gamma, beta): return gamma * x + beta

model = LatentDecisionGatesUnifiedModule().to(device)
with torch.no_grad():
    model.linear.weight.copy_(weights['weight'].t()); model.linear.bias.copy_(weights['bias'])
print('LatentDecisionGatesUnifiedModule initialized successfully.')

In [ ]:
err_hw = (model.highway(weights['highway_x'], weights['w_h'], weights['b_h'], weights['w_t'], weights['b_t']) - weights['expected_highway']).abs().max().item()
assert err_hw < 1e-5, f'Highway parity error: {err_hw:.2e}'

err_glu = (model.glu(weights['glu_x'], weights['w_glu'], weights['b_glu']) - weights['expected_glu']).abs().max().item()
assert err_glu < 1e-5, f'GLU parity error: {err_glu:.2e}'

hc_g, hc_l0 = model.hard_concrete(weights['log_alpha'], u=weights['hard_concrete_u'])
err_hc_g = (hc_g - weights['expected_hard_concrete_g']).abs().max().item()
err_hc_l0 = (hc_l0 - weights['expected_hard_concrete_l0']).abs().max().item()
assert err_hc_g < 1e-5 and err_hc_l0 < 1e-5, f'Hard concrete error: g={err_hc_g:.2e}, l0={err_hc_l0:.2e}'

err_spm = (model.sparsemax(weights['sparsemax_logits']) - weights['expected_sparsemax']).abs().max().item()
assert err_spm < 1e-5, f'Sparsemax parity error: {err_spm:.2e}'

p_pnd, steps_pnd, kl_pnd = model.pondernet(weights['ponder_lambdas'])
err_p = (p_pnd - weights['expected_ponder_p']).abs().max().item()
err_st = (steps_pnd - weights['expected_ponder_steps']).abs().max().item()
err_kl = (kl_pnd - weights['expected_ponder_kl']).abs().max().item()
assert err_p < 1e-5 and err_st < 1e-5 and err_kl < 1e-5, f'PonderNet error: p={err_p:.2e}, steps={err_st:.2e}, kl={err_kl:.2e}'

err_film = (model.film(weights['film_x'], weights['film_gamma'], weights['film_beta']) - weights['expected_film']).abs().max().item()
assert err_film < 1e-5, f'FiLM parity error: {err_film:.2e}'

err_anchor = (model(test_in) - test_out).abs().max().item()
assert err_anchor < 1e-5, f'Anchor parity error: {err_anchor:.2e}'

print(f'All 7 invariants verified! Highway: {err_hw:.2e}, GLU: {err_glu:.2e}, HardConcrete: {err_hc_g:.2e}/{err_hc_l0:.2e}, Sparsemax: {err_spm:.2e}, PonderNet: {err_p:.2e}/{err_st:.2e}/{err_kl:.2e}, FiLM: {err_film:.2e}, Anchor: {err_anchor:.2e}')

In [ ]:
track_fns = {
    'Linear Anchor': lambda: model(test_in),
    'Highway Network': lambda: model.highway(weights['highway_x'], weights['w_h'], weights['b_h'], weights['w_t'], weights['b_t']),
    'GLU': lambda: model.glu(weights['glu_x'], weights['w_glu'], weights['b_glu']),
    'Hard Concrete': lambda: model.hard_concrete(weights['log_alpha'], u=weights['hard_concrete_u']),
    'Sparsemax': lambda: model.sparsemax(weights['sparsemax_logits']),
    'PonderNet': lambda: model.pondernet(weights['ponder_lambdas']),
    'FiLM': lambda: model.film(weights['film_x'], weights['film_gamma'], weights['film_beta']),
}
bench_results = core.benchmark.compare_tracks(track_fns, warmup=3, iters=15, batch_size=4)
print(core.benchmark.format_benchmark_table(bench_results))

In [ ]:
# Active Failure Mode: Dense Softmax Support Leakage vs Sparsemax Exact-Zero Truncation
# When routing over discrete experts or selecting features, softmax assigns strictly positive
# probability mass to highly negative, irrelevant logits, leaking noise and executing 100% of branches!
sample_logits = torch.tensor([[3.0, 1.0, 0.0, -8.0]], device=device)
p_softmax = torch.softmax(sample_logits, dim=-1)
p_sparsemax = model.sparsemax(sample_logits)

print(f'[FAILURE DEMONSTRATED] Softmax non-zero mass on negative outlier: p[-1] = {p_softmax[0, -1].item():.4e} > 0.0!')
assert (p_softmax > 0.0).all().item(), 'Expected dense softmax support'

# Sparsemax Euclidean projection truncates suboptimal logits to exact zero:
print(f'[STABILIZED] Sparsemax exact zero truncation: p[-1] = {p_sparsemax[0, -1].item():.1f} | Sum = {p_sparsemax.sum().item():.1f}')
assert p_sparsemax[0, -1].item() == 0.0, f'Expected exact zero, got {p_sparsemax[0, -1].item()}'
assert abs(p_sparsemax.sum().item() - 1.0) < 1e-5, 'Sparsemax must sum to 1.0'
assert torch.allclose(model(test_in), test_out, atol=1e-5), 'Linear anchor mismatch'
print('[CORRECT] Latent decision gates and exact-zero sparse projection verified successfully.')

## 8. Algorithmic Takeaways & Cross-Domain Isomorphisms

1. **Identity Precedence in Deep Gating:** Highway transform gate biases must be initialized negative ($b_T < 0$) to bias early training toward identity skip connections, preventing vanishing gradient signals.
2. **Exact Zeros Enable Conditional Execution:** Hard Concrete ($L_0$) and Sparsemax output exact `0.0` values, permitting true conditional branch pruning and sparse feature selection without noise leakage.
3. **Principled Halting Dynamics:** PonderNet's geometric prior KL penalty prevents the step-collapse pathology inherent to Graves' linear ACT ponder penalty.

---
**Curriculum Navigation:** [Master Index](file:///c:/Users/Liam/Code/AI-Examples/README.md) | [Previous: 11_losses_proper_scoring_evidential_and_differentiable_ranking](file:///c:/Users/Liam/Code/AI-Examples/notebooks/03_building_blocks_and_losses/11_losses_proper_scoring_evidential_and_differentiable_ranking.ipynb)